In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# load CSVs and parse timestamps as UTC datetimes
df_metric = pd.read_csv("metric.csv")
df_metric["timestamp"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)

df_trace = pd.read_csv("trace.csv")
df_trace["timestamp"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)

df_log = pd.read_csv("log.csv")
df_log["timestamp"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)

df_errors = pd.read_csv("error_logs.csv")
df_errors["timestamp"] = pd.to_datetime(df_errors["timestamp"], unit="s", utc=True)

# Aggregate metrics by (cmdb_id, kpi_name): count, mean(value), global P95(value)
metric_summary = (
    df_metric
    .groupby(["cmdb_id", "kpi_name"])["value"]
    .agg(count="count", mean="mean", p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# format numeric columns compactly
metric_summary["mean"] = metric_summary["mean"].round(6)
metric_summary["p95"] = metric_summary["p95"].round(6)

# top 20 groups by count descending
metric_top20 = metric_summary.sort_values("count", ascending=False).loc[:, ["cmdb_id", "kpi_name", "count", "mean", "p95"]].head(20)

# Trace: unique trace_name values with counts (top 20)
trace_counts = (
    df_trace
    .groupby("trace_name")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

# Log: unique log_name values with counts (top 20)
log_counts = (
    df_log
    .groupby("log_name")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

# Error logs: up to first 5 rows (all columns)
errors_head = df_errors.head(5)

# Display compact results
metric_top20, trace_counts, log_counts, errors_head

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregation results:\n"
    "- Metrics: Top groups (by sample count) are for 'carts' and 'carts-db' services. "
    "Each reported metric group has 25 samples. Notable examples: 'carts' shows CPU mean ~1.46, memory mean ~2.12e8, "
    "socket mean ~9.92 and workload mean ~8.22. 'carts-db' shows CPU mean ~4.43, diskio mean ~2.08e6, memory mean ~8.48e7. "
    "'catalogue' and 'catalogue-db' also appear but with smaller CPU/workload values in comparison.\n"
    "- Traces: No trace features were present (trace_name list is empty).\n"
    "- Logs: Two log metrics exist: 'log.error_count' and 'log.row_count', each with 228 samples.\n"
    "- Error logs: No error log rows were returned (empty).\n\n"
    "Plain conclusion: the telemetry shows the highest activity/metric values concentrated on 'carts' and 'carts-db'. "
    "There are no trace records and no raw error log entries to inspect further. Logs show aggregated counts but no error log details."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count          mean           p95
0          carts         cpu     25  1.460176e+00  1.616533e+00
1          carts  latency-50     25  1.777300e-02  1.852000e-02
2          carts  latency-90     25  3.932200e-02  4.103200e-02
3          carts         mem     25  2.116232e+08  2.127850e+08
4          carts      socket     25  9.921080e+00  1.075000e+01
5          carts    workload     25  8.218737e+00  8.591573e+00
6       carts-db         cpu     25  4.426165e+00  5.031349e+00
7       carts-db      diskio     25  2.083193e+06  2.380252e+06
8       carts-db         mem     25  8.475230e+07  8.694617e+07
9       carts-db      socket     25  6.000000e+00  6.000000e+00
10     catalogue         cpu     25  1.693450e-01  1.846590e-01
11     catalogue  latency-50     25  3.031000e-03  3.056000e-03
12     catalogue  latency-90     25  4.655000e-03  4.701000e-03
13     catalogue         mem     25  6.173824e+06  6.233580e+06
14     catalogue      socket     25  5.000000e+00  5.000000e+00
15     catalogue    workload     25  4.072708e+00  4.238007e+00
16  catalogue-db         cpu     25  2.030100e-01  2.145350e-01
17  catalogue-db      diskio     25  0.000000e+00  0.000000e+00
18  catalogue-db         mem     25  4.305086e+08  4.305101e+08
19  catalogue-db      socket     25  4.000000e+00  4.000000e+00, Empty DataFrame
Columns: [trace_name, count]
Index: [],           log_name  count
0  log.error_count    228
1    log.row_count    228, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing dataframes: df_metric, df_log, df_trace, df_errors
# (They were loaded previously with timestamps parsed as UTC)

# Incident window (UTC)
start = pd.to_datetime("2024-01-20 12:40:57", utc=True)
end   = pd.to_datetime("2024-01-20 13:10:57", utc=True)

# 1) Metrics
# Global P95 per (cmdb_id, kpi_name) using entire series
metric_p95_global = (
    df_metric
    .groupby(["cmdb_id", "kpi_name"])["value"]
    .quantile(0.95)
    .reset_index(name="p95")
)

# Filter metric rows to incident window
df_metric_win = df_metric[(df_metric["timestamp"] >= start) & (df_metric["timestamp"] <= end)].copy()

# Aggregate stats within window per group
metric_win_agg = (
    df_metric_win
    .groupby(["cmdb_id", "kpi_name"])
    .agg(
        count_in_window=("value", "count"),
        earliest_timestamp_in_window=("timestamp", "min"),
        latest_timestamp_in_window=("timestamp", "max"),
        max_value_in_window=("value", "max"),
    )
    .reset_index()
)

# Merge with global p95
metric_merged = metric_win_agg.merge(metric_p95_global, on=["cmdb_id", "kpi_name"], how="left")

# Compute flags and severity ratio (inf if p95 == 0)
metric_merged["max_over_p95_flag"] = metric_merged["max_value_in_window"] >= metric_merged["p95"]
metric_merged["severity_ratio"] = np.where(
    metric_merged["p95"] == 0,
    np.inf,
    metric_merged["max_value_in_window"] / metric_merged["p95"]
)

# Filter to groups where max_over_p95_flag is True, sort by severity_ratio desc, limit 20
metric_anomalies = (
    metric_merged[metric_merged["max_over_p95_flag"]]
    .sort_values("severity_ratio", ascending=False)
    .loc[
        :,
        [
            "cmdb_id",
            "kpi_name",
            "p95",
            "count_in_window",
            "earliest_timestamp_in_window",
            "latest_timestamp_in_window",
            "max_value_in_window",
            "max_over_p95_flag",
            "severity_ratio",
        ],
    ]
    .head(20)
)

# 2) Logs
# Global P95 and P5 per (cmdb_id, log_name) using entire series
log_quantiles_global = (
    df_log
    .groupby(["cmdb_id", "log_name"])["value"]
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# Filter log rows to incident window
df_log_win = df_log[(df_log["timestamp"] >= start) & (df_log["timestamp"] <= end)].copy()

# Aggregate stats within window per group
log_win_agg = (
    df_log_win
    .groupby(["cmdb_id", "log_name"])
    .agg(
        count_in_window=("value", "count"),
        earliest_timestamp_in_window=("timestamp", "min"),
        latest_timestamp_in_window=("timestamp", "max"),
        max_value_in_window=("value", "max"),
        min_value_in_window=("value", "min"),
    )
    .reset_index()
)

# Merge with global quantiles
log_merged = log_win_agg.merge(log_quantiles_global, on=["cmdb_id", "log_name"], how="left")

# Flags
log_merged["max_over_p95_flag"] = log_merged["max_value_in_window"] >= log_merged["p95"]
log_merged["min_below_p5_flag"] = log_merged["min_value_in_window"] <= log_merged["p5"]

# Compute sort key (p95 - min_value_in_window); handle NaN
log_merged["p95_minus_min"] = log_merged["p95"] - log_merged["min_value_in_window"]

# Filter groups where either flag is True
log_anomalies = (
    log_merged[(log_merged["max_over_p95_flag"]) | (log_merged["min_below_p5_flag"])]
    .sort_values(by=["max_over_p95_flag", "p95_minus_min"], ascending=[False, False])
    .loc[
        :,
        [
            "cmdb_id",
            "log_name",
            "p95",
            "p5",
            "count_in_window",
            "earliest_timestamp_in_window",
            "latest_timestamp_in_window",
            "max_value_in_window",
            "min_value_in_window",
            "max_over_p95_flag",
            "min_below_p5_flag",
        ],
    ]
    .head(20)
)

# 3) Error logs: up to first 5 rows in the incident window (timestamp, cmdb_id, message)
df_errors_win = df_errors[(df_errors["timestamp"] >= start) & (df_errors["timestamp"] <= end)].copy()
if df_errors_win.shape[0] > 0:
    errors_in_window_head = df_errors_win.loc[:, ["timestamp", "cmdb_id", "message"]].head(5)
else:
    # empty indicator DataFrame with expected columns
    errors_in_window_head = pd.DataFrame(columns=["timestamp", "cmdb_id", "message"])

# 4) Trace: count rows in incident window
trace_count_in_window = df_trace[(df_trace["timestamp"] >= start) & (df_trace["timestamp"] <= end)].shape[0]

# Final compact outputs
metric_anomalies, log_anomalies, errors_in_window_head, trace_count_in_window

```
Out[2]:
```
summary = (
    "Summary of incident-window analysis (2024-01-20 12:40:57 — 2024-01-20 13:10:57 UTC):\n"
    "- Metrics: Multiple (service, KPI) groups exceeded their global P95 in the window. "
    "Several groups have global P95==0 which yields infinite severity ratios (these include orders.error, payment.diskio, front-end.error, catalogue-db.diskio, rabbitmq.diskio) — some of those with p95==0 also have max==0 (not informative). "
    "Among the finite-severity cases the largest deviations were queue-master.cpu (severity ~2.82), shipping.cpu (~2.10), orders.cpu (~1.99), queue-master.diskio (~1.31), rabbitmq-exporter.cpu (~1.24), and payment latency-90 (~1.11). "
    "- Logs: Many services show log anomalies (row_count and error_count). Notable affected services with both high peaks and low troughs include front-end, user, queue-master, catalogue, orders, payment, carts and shipping — e.g., front-end log.row_count exceeded its p95 and also dropped below its p5 in the window. Several log.error_count groups have p95==0 with all-zero values (flagged but not actionable).\n"
    "- Error logs: No error_log entries were found in the incident window.\n"
    "- Traces: No trace rows were present in the window (count = 0).\n\n"
    "Actionable lead: focus on queue-master, shipping, orders, and payment (and front-end log.row_count) for root-cause investigation given the strongest metric and log deviations during the window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name           p95  count_in_window earliest_timestamp_in_window latest_timestamp_in_window  max_value_in_window  max_over_p95_flag  severity_ratio
28             orders       error  0.000000e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         0.000000e+00               True             inf
39            payment      diskio  0.000000e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         8.043658e+04               True             inf
21          front-end       error  0.000000e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         0.000000e+00               True             inf
17       catalogue-db      diskio  0.000000e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         0.000000e+00               True             inf
50           rabbitmq      diskio  0.000000e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         0.000000e+00               True             inf
45       queue-master         cpu  8.171286e-01               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         2.300657e+00               True        2.815539
60           shipping         cpu  1.296582e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         2.719597e+00               True        2.097512
27             orders         cpu  1.868077e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         3.719276e+00               True        1.990965
46       queue-master      diskio  4.946849e+04               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         6.480865e+04               True        1.310100
53  rabbitmq-exporter         cpu  1.605985e-04               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         1.992906e-04               True        1.240925
41            payment  latency-90  4.111080e-02               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         4.549107e-02               True        1.106548
30             orders  latency-90  7.216342e-02               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         7.800485e-02               True        1.080947
7            carts-db      diskio  2.380252e+06               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         2.572472e+06               True        1.080756
5               carts    workload  8.591573e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         9.267000e+00               True        1.078615
62           shipping  latency-90  5.950223e-03               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         6.364184e-03               True        1.069571
64           shipping      socket  6.186667e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         6.583333e+00               True        1.064116
0               carts         cpu  1.616533e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         1.704205e+00               True        1.054234
25          front-end      socket  8.590000e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         9.000000e+00               True        1.047730
6            carts-db         cpu  5.031349e+00               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         5.261939e+00               True        1.045831
32             orders      socket  1.740333e+01               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00         1.818333e+01               True        1.044819,          cmdb_id         log_name     p95      p5  count_in_window earliest_timestamp_in_window latest_timestamp_in_window  max_value_in_window  min_value_in_window  max_over_p95_flag  min_below_p5_flag
7      front-end    log.row_count  2080.6  1822.6               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                 2117                   95               True               True
21          user    log.row_count   617.0   560.2               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                  638                   35               True               True
15  queue-master    log.row_count   576.0   513.0               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                  612                   36               True               True
5      catalogue    log.row_count   171.2   148.6               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                  175                    8               True               True
9         orders    log.row_count   128.0   114.0               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                  136                    8               True               True
13       payment    log.row_count   104.0    97.0               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                  108                    6               True               True
1          carts    log.row_count   100.0    25.0               23    2024-01-20 12:44:00+00:00  2024-01-20 13:07:00+00:00                  125                   25               True               True
19      shipping    log.row_count    64.0    57.0               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                   68                    4               True               True
0          carts  log.error_count     0.0     0.0               23    2024-01-20 12:44:00+00:00  2024-01-20 13:07:00+00:00                    0                    0               True               True
2       carts-db  log.error_count     0.0     0.0               24    2024-01-20 12:44:00+00:00  2024-01-20 13:07:00+00:00                    0                    0               True               True
3       carts-db    log.row_count     1.0     1.0               24    2024-01-20 12:44:00+00:00  2024-01-20 13:07:00+00:00                    2                    1               True               True
4      catalogue  log.error_count     0.0     0.0               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                    0                    0               True               True
6      front-end  log.error_count     0.0     0.0               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                    0                    0               True               True
8         orders  log.error_count     0.0     0.0               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                    0                    0               True               True
10     orders-db  log.error_count     0.0     0.0                1    2024-01-20 12:50:00+00:00  2024-01-20 12:50:00+00:00                    0                    0               True               True
11     orders-db    log.row_count     2.0     2.0                1    2024-01-20 12:50:00+00:00  2024-01-20 12:50:00+00:00                    2                    2               True               True
12       payment  log.error_count     0.0     0.0               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                    0                    0               True               True
14  queue-master  log.error_count     0.0     0.0               25    2024-01-20 12:43:00+00:00  2024-01-20 13:07:00+00:00                    0                    0               True               True
16    session-db  log.error_count     0.0     0.0                5    2024-01-20 12:46:00+00:00  2024-01-20 13:06:00+00:00                    0                    0               True               True
17    session-db    log.row_count     5.0     5.0                5    2024-01-20 12:46:00+00:00  2024-01-20 13:06:00+00:00                    5                    5               True               True, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], 0)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse incident window and preloaded dataframes from previous steps
start = pd.to_datetime("2024-01-20 12:40:57", utc=True)
end   = pd.to_datetime("2024-01-20 13:10:57", utc=True)

# Ensure windowed dataframes exist (created previously); if not, create them
df_metric_win = df_metric[(df_metric["timestamp"] >= start) & (df_metric["timestamp"] <= end)].copy()
df_log_win = df_log[(df_log["timestamp"] >= start) & (df_log["timestamp"] <= end)].copy()

# Helpers: global thresholds (precomputed earlier)
# metric_p95_global: columns ['cmdb_id','kpi_name','p95']
# log_quantiles_global: columns ['cmdb_id','log_name','p95','p5']

# Candidate list
candidates = [
    ("queue-master", "cpu", "metric"),
    ("queue-master", "diskio", "metric"),
    ("shipping", "cpu", "metric"),
    ("orders", "cpu", "metric"),
    ("payment", "latency-90", "metric"),
    ("front-end", "log.row_count", "log"),
]

rows = []
for cmdb_id, kpi_name, kind in candidates:
    if kind == "metric":
        # get global p95
        row_thresh = metric_p95_global[
            (metric_p95_global["cmdb_id"] == cmdb_id) & (metric_p95_global["kpi_name"] == kpi_name)
        ]
        p95 = row_thresh["p95"].iloc[0] if not row_thresh.empty else np.nan

        dfg = df_metric_win[(df_metric_win["cmdb_id"] == cmdb_id) & (df_metric_win["kpi_name"] == kpi_name)].sort_values("timestamp")
        if dfg.empty or np.isnan(p95):
            rows.append({
                "cmdb_id": cmdb_id,
                "kpi_name": kpi_name,
                "p95": p95,
                "earliest_anomaly_timestamp_in_window": pd.NaT,
                "number_of_consecutive_anomalous_points_at_earliest_run": 0,
                "total_anomalous_points_in_window": 0,
                "max_value_in_window": 0.0,
                "severity_ratio": 0.0,
            })
            continue

        vals = dfg["value"].values
        # anomaly condition: value >= p95 (p95 may be 0)
        anomaly_mask = vals >= p95

        if not anomaly_mask.any():
            rows.append({
                "cmdb_id": cmdb_id,
                "kpi_name": kpi_name,
                "p95": p95,
                "earliest_anomaly_timestamp_in_window": pd.NaT,
                "number_of_consecutive_anomalous_points_at_earliest_run": 0,
                "total_anomalous_points_in_window": 0,
                "max_value_in_window": 0.0,
                "severity_ratio": 0.0,
            })
            continue

        # earliest anomaly timestamp
        first_idx = int(np.argmax(anomaly_mask))
        earliest_ts = dfg.iloc[first_idx]["timestamp"]

        # count consecutive True from first_idx
        consec = 0
        for v in anomaly_mask[first_idx:]:
            if v:
                consec += 1
            else:
                break

        total_anom = int(anomaly_mask.sum())
        max_val = float(dfg["value"].max())
        severity = np.inf if p95 == 0 else max_val / p95

        rows.append({
            "cmdb_id": cmdb_id,
            "kpi_name": kpi_name,
            "p95": p95,
            "earliest_anomaly_timestamp_in_window": earliest_ts,
            "number_of_consecutive_anomalous_points_at_earliest_run": int(consec),
            "total_anomalous_points_in_window": total_anom,
            "max_value_in_window": max_val,
            "severity_ratio": severity,
        })

    else:  # log
        # get global p95 and p5
        row_thresh = log_quantiles_global[
            (log_quantiles_global["cmdb_id"] == cmdb_id) & (log_quantiles_global["log_name"] == kpi_name)
        ]
        if not row_thresh.empty:
            p95 = row_thresh["p95"].iloc[0]
            p5 = row_thresh["p5"].iloc[0]
        else:
            p95 = np.nan
            p5 = np.nan

        dfg = df_log_win[(df_log_win["cmdb_id"] == cmdb_id) & (df_log_win["log_name"] == kpi_name)].sort_values("timestamp")
        if dfg.empty or (np.isnan(p95) and np.isnan(p5)):
            rows.append({
                "cmdb_id": cmdb_id,
                "kpi_name": kpi_name,
                "p95": p95,
                "earliest_anomaly_timestamp_in_window": pd.NaT,
                "number_of_consecutive_anomalous_points_at_earliest_run": 0,
                "total_anomalous_points_in_window": 0,
                "max_value_in_window": 0.0,
                "severity_ratio": 0.0,
            })
            continue

        vals = dfg["value"].values
        # anomaly condition for logs: value >= p95 OR value <= p5
        # handle NaN thresholds: if p95 is NaN, only p5 used and vice versa
        if np.isnan(p95):
            anomaly_mask = vals <= p5
        elif np.isnan(p5):
            anomaly_mask = vals >= p95
        else:
            anomaly_mask = (vals >= p95) | (vals <= p5)

        if not anomaly_mask.any():
            rows.append({
                "cmdb_id": cmdb_id,
                "kpi_name": kpi_name,
                "p95": p95,
                "earliest_anomaly_timestamp_in_window": pd.NaT,
                "number_of_consecutive_anomalous_points_at_earliest_run": 0,
                "total_anomalous_points_in_window": 0,
                "max_value_in_window": 0.0,
                "severity_ratio": 0.0,
            })
            continue

        first_idx = int(np.argmax(anomaly_mask))
        earliest_ts = dfg.iloc[first_idx]["timestamp"]

        consec = 0
        for v in anomaly_mask[first_idx:]:
            if v:
                consec += 1
            else:
                break

        total_anom = int(anomaly_mask.sum())
        max_val = float(dfg["value"].max())
        severity = np.inf if p95 == 0 else (max_val / p95 if not np.isnan(p95) else np.inf)

        rows.append({
            "cmdb_id": cmdb_id,
            "kpi_name": kpi_name,
            "p95": p95,
            "earliest_anomaly_timestamp_in_window": earliest_ts,
            "number_of_consecutive_anomalous_points_at_earliest_run": int(consec),
            "total_anomalous_points_in_window": total_anom,
            "max_value_in_window": max_val,
            "severity_ratio": severity,
        })

results_df = pd.DataFrame(rows, columns=[
    "cmdb_id",
    "kpi_name",
    "p95",
    "earliest_anomaly_timestamp_in_window",
    "number_of_consecutive_anomalous_points_at_earliest_run",
    "total_anomalous_points_in_window",
    "max_value_in_window",
    "severity_ratio",
])

results_df

```
Out[3]:
```
Summary of anomalies in the incident window (2024-01-20 12:40:57 — 2024-01-20 13:10:57 UTC):

- queue-master, cpu — p95=0.817129. Earliest anomaly: 2024-01-20 12:44:00 UTC. First consecutive anomalous run length: 1. Total anomalous points in window: 2. Max=2.300657. Severity ratio=2.8155.

- queue-master, diskio — p95=49468.490795. Earliest anomaly: 2024-01-20 12:45:00 UTC. First run length: 1. Total anomalous points: 2. Max=64808.646929. Severity ratio=1.3101.

- shipping, cpu — p95=1.296582. Earliest anomaly: 2024-01-20 12:58:00 UTC. First run length: 2. Total anomalous points: 2. Max=2.719597. Severity ratio=2.0975.

- orders, cpu — p95=1.868077. Earliest anomaly: 2024-01-20 12:44:00 UTC. First run length: 1. Total anomalous points: 2. Max=3.719276. Severity ratio=1.9910.

- payment, latency-90 — p95=0.041111. Earliest anomaly: 2024-01-20 13:00:00 UTC. First run length: 2. Total anomalous points: 2. Max=0.045491. Severity ratio=1.1065.

- front-end, log.row_count — p95=2080.6. Earliest anomaly: 2024-01-20 12:43:00 UTC. First run length: 1. Total anomalous points: 4. Max=2117.0. Severity ratio=1.0175.

Interpretation: All six candidates showed anomalous points during the window; the largest relative deviations (severity ratio) are queue-master.cpu, shipping.cpu, and orders.cpu. front-end log.row_count shows small exceedance above its p95 and multiple anomalous points.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id       kpi_name           p95 earliest_anomaly_timestamp_in_window  number_of_consecutive_anomalous_points_at_earliest_run  total_anomalous_points_in_window  max_value_in_window  severity_ratio
0  queue-master            cpu      0.817129            2024-01-20 12:44:00+00:00                                                  1                                      2             2.300657        2.815539
1  queue-master         diskio  49468.490795            2024-01-20 12:45:00+00:00                                                  1                                      2         64808.646929        1.310100
2      shipping            cpu      1.296582            2024-01-20 12:58:00+00:00                                                  2                                      2             2.719597        2.097512
3        orders            cpu      1.868077            2024-01-20 12:44:00+00:00                                                  1                                      2             3.719276        1.990965
4       payment     latency-90      0.041111            2024-01-20 13:00:00+00:00                                                  2                                      2             0.045491        1.106548
5     front-end  log.row_count   2080.600000            2024-01-20 12:43:00+00:00                                                  1                                      4          2117.000000        1.017495```
```